# Task 5 — Kaggle probe 2: a 25% alert budget

The user-reported overall leaderboard score rose from **0.72700 (v6, about 5% alerts)** to **0.74446 (v8, about 15% alerts)**. This notebook tests whether raising the alert budget further to 25% helps the same saved engineered-feature RBF One-Class SVM. Tasks 1–4 and the confirmed `1 = anomaly` orientation remain fixed. No Test labels or offline F3 are available.

The lecturer reportedly permits unsupervised fitting on the provided Task 5 Test file; the written competition rule still differs. This experiment changes only a cutoff on the saved detector ranking. The controlled overall-score response will be available only after the user submits the prepared CSV.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
from hashlib import sha256
import copy,json,sys,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np,pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.pipeline.build_sample_submission import build_sample_submission
SEED=set_seed()
DATA_NAME='eda_1_v2'
BASE_MODEL='t5_r2_ocsvm_nu10_engineered_v2'
MODEL_NAME='t5_r4_ocsvm_nu10_alert25_v4'
SUBMISSION_NAME='kaggle_t12345_t5_alert25_v9'
BASE_SUBMISSION='kaggle_t12345_gate_resnet_v6'
data_card=json.loads((SRC/'task5'/'data'/'eda'/f'{DATA_NAME}.json').read_text(encoding='utf8'))
assert data_card['name']==DATA_NAME and data_card['task']=='task5' and set(data_card['splits'])=={'test'}
test_path=SRC/data_card['splits']['test']['path']
test=pd.read_parquet(test_path) if data_card['format']=='parquet' else pd.read_csv(test_path)
assert test.player_id.is_unique and 'is_anomaly' not in test
X=test.drop(columns='player_id')
base_model_card=json.loads((SRC/'task5'/'save'/f'{BASE_MODEL}.json').read_text(encoding='utf8'))
assert base_model_card['data_name']==DATA_NAME
base_model=joblib.load(SRC/base_model_card['model_path'])
base_submission_card=json.loads((SRC/'submission'/f'{BASE_SUBMISSION}.json').read_text(encoding='utf8'))
base_submission_path=SRC/base_submission_card['submission_path']
assert sha256(base_submission_path.read_bytes()).hexdigest()==base_submission_card['sha256']
base_submission=pd.read_csv(base_submission_path)
PRIOR_SUBMISSION='kaggle_t12345_t5_alert15_v8'
prior_card=json.loads((SRC/'submission'/f'{PRIOR_SUBMISSION}.json').read_text(encoding='utf8'))
prior_path=SRC/prior_card['submission_path']
assert sha256(prior_path.read_bytes()).hexdigest()==prior_card['sha256']
prior_submission=pd.read_csv(prior_path)
sample=pd.read_csv(SRC/'submission'/'sample_submission.csv')
assert base_submission.columns.tolist()==sample.columns.tolist()==['id','task1','task2','task3','task4','task5']
assert len(test)==len(base_submission)==len(sample)==25889
assert base_submission.id.tolist()==sample.id.tolist() and sample.id.is_unique
assert prior_submission[['id','task1','task2','task3','task4']].equals(base_submission[['id','task1','task2','task3','task4']])
assert prior_submission.task5.sum()==3971
base_prediction=np.asarray(base_model.predict(X),dtype=int)
assert np.array_equal(base_prediction,base_submission.task5.to_numpy(dtype=int)), 'Saved model must reproduce v6 Task 5 exactly'
assert set(base_prediction)=={0,1}
print('Verified v6 Task 5 reconstruction:',int(base_prediction.sum()),'alerts /',len(base_prediction))

Verified v6 Task 5 reconstruction: 1365 alerts / 25889


## Fit and save the cutoff variant

Use the same fitted OCSVM and the same 12,000-row fit split. Only its cutoff changes to the 75th percentile. As in the previous probe, an overlapping 10,000-row fit checks top-25% rank stability on an unlabeled 4,000-row diagnostic holdout; this proxy is not F3.

In [2]:
rows=np.arange(len(X))
dev_idx,hold_idx=train_test_split(rows,train_size=12000,test_size=4000,random_state=SEED)
dev_b_idx=np.random.default_rng(SEED).choice(dev_idx,size=10000,replace=False)
SAVE=SRC/'task5'/'save'
model_path,model_card_path,eval_path=(SAVE/f'{MODEL_NAME}.joblib',SAVE/f'{MODEL_NAME}.json',SAVE/f'{MODEL_NAME}-evaluation.json')
if not (model_path.exists() and model_card_path.exists() and eval_path.exists()):
    assert not any(path.exists() for path in (model_path,model_card_path,eval_path)), 'Inspect incomplete artifact before rerun'
    candidate=copy.deepcopy(base_model)
    candidate.anomaly_fraction=.25
    candidate.threshold_=float(np.quantile(base_model.anomaly_score(X.iloc[dev_idx]),.75))
    assert np.array_equal(candidate.anomaly_score(X.iloc[hold_idx]),base_model.anomaly_score(X.iloc[hold_idx]))
    alternate=clone(base_model).fit(X.iloc[dev_b_idx])
    a=base_model.anomaly_score(X.iloc[hold_idx]);b=alternate.anomaly_score(X.iloc[hold_idx])
    k=round(.25*len(hold_idx))
    top_a=set(np.argpartition(a,-k)[-k:]);top_b=set(np.argpartition(b,-k)[-k:])
    stability=len(top_a & top_b)/len(top_a | top_b)
    save_model(task=5,name=MODEL_NAME,model=candidate,data_name=DATA_NAME,
        card={'idea':'Reuse saved OCSVM ranking with 25% training-fit alert cutoff',
              'base_model':BASE_MODEL,'alert_budget':.25,'fit_rows':12000,'seed':SEED,
              'scope':'user-reported lecturer permission for unsupervised Task 5 Test fit'})
    result=evaluate_model(task=5,name=MODEL_NAME,score=stability,
        metric_name='top25_rank_stability_jaccard',higher_is_better=True,
        card={'meaning':'Top-25% rank overlap between 12k and overlapping 10k fits on 4k unlabeled rows; not F3',
              'seed':SEED})
    print('Top-25% stability proxy:',result['score'])
else:
    saved_card=json.loads(model_card_path.read_text(encoding='utf8'))
    assert saved_card['data_name']==DATA_NAME and saved_card['card']['base_model']==BASE_MODEL
    candidate=joblib.load(model_path)
    print('Loaded existing model and proxy:',json.loads(eval_path.read_text(encoding='utf8'))['score'])
assert np.array_equal(candidate.anomaly_score(X),base_model.anomaly_score(X))
assert candidate.threshold_ < base_model.threshold_
print('Cutoffs: v6',base_model.threshold_,'candidate',candidate.threshold_)

Top-25% stability proxy: 0.9821605550049554


Cutoffs: v6 -556.3941659439402 candidate -582.2007828999981


## Compare the new alerts with v6 and v8

The 25% set must contain all alerts from both earlier cutoffs. Report both added-alert counts. The security flags are descriptive model inputs, not anomaly labels.

In [3]:
candidate_prediction=np.asarray(candidate.predict(X),dtype=int)
assert set(candidate_prediction)=={0,1}
assert np.all(candidate_prediction>=base_prediction)
prior_prediction=prior_submission.task5.to_numpy(dtype=int)
assert np.all(candidate_prediction>=prior_prediction)
new_vs_prior=(candidate_prediction==1)&(prior_prediction==0)
new_alert=(candidate_prediction==1)&(base_prediction==0)
print('Alert counts:',{'v6':int(base_prediction.sum()),'candidate':int(candidate_prediction.sum()),'new_since_v6':int(new_alert.sum()),'new_since_v8':int(new_vs_prior.sum())})
print('Candidate alert fraction:',candidate_prediction.mean())
flag_cols=['password_changed','vpn_usage','suspicious_login_time','mass_item_sale']
flags=X[flag_cols].fillna(0).gt(0).any(axis=1).to_numpy()
for name,mask in [('all',np.ones(len(X),dtype=bool)),('v6 alerts',base_prediction==1),('new since v8',new_vs_prior)]:
    print(name,'rows',int(mask.sum()),'security flag rate',float(flags[mask].mean()))
display(pd.DataFrame({'group':['v6 alerts','new alerts','still unalerted'],
    'rows':[int(base_prediction.sum()),int(new_alert.sum()),int((candidate_prediction==0).sum())]}))

Alert counts: {'v6': 1365, 'candidate': 6531, 'new_since_v6': 5166, 'new_since_v8': 2560}
Candidate alert fraction: 0.25226930356522076
all rows 25889 security flag rate 0.25443238441036736
v6 alerts rows 1365 security flag rate 0.5135531135531135
new since v8 rows 2560 security flag rate 0.2578125


,group,rows
0,v6 alerts,1365
1,new alerts,5166
2,still unalerted,19358


## Build the controlled Kaggle draft

The per-task file comes from `build_sample_submission`. The final CSV maps Test row order to the established ANS ID order and copies Tasks 1–4 from scored v6. Its card records changes against both v6 and v8. No Kaggle upload occurs here.

In [4]:
TASK5_DRAFT=MODEL_NAME
part_path=SRC/'task5'/'submission'/f'{TASK5_DRAFT}.csv'
part_card_path=SRC/'task5'/'submission'/f'{TASK5_DRAFT}.json'
if not (part_path.exists() and part_card_path.exists()):
    assert not (part_path.exists() or part_card_path.exists()),'Inspect incomplete per-task submission'
    build_sample_submission(task=5,name=TASK5_DRAFT,model_name=MODEL_NAME,id_column='player_id',
        card={'idea':'25% alert-budget Task 5 probe','scope':'Test row order mapped to ANS IDs in final assembly'})
part=pd.read_csv(part_path,sep='|')
assert part.columns.tolist()==['id','task5'] and part.id.tolist()==test.player_id.tolist()
assert np.array_equal(part.task5.to_numpy(dtype=int),candidate_prediction)
output=base_submission.copy()
output['task5']=candidate_prediction
assert output[['id','task1','task2','task3','task4']].equals(base_submission[['id','task1','task2','task3','task4']])
assert output.id.tolist()==sample.id.tolist() and output.notna().all().all()
assert np.array_equal(output.task5.to_numpy(dtype=int),candidate_prediction)
assert int((output.task5!=base_submission.task5).sum())==int(new_alert.sum())
out_path=SRC/'submission'/f'{SUBMISSION_NAME}.csv'
out_card_path=SRC/'submission'/f'{SUBMISSION_NAME}.json'
if not (out_path.exists() and out_card_path.exists()):
    assert not (out_path.exists() or out_card_path.exists()),'Inspect incomplete final submission'
    output.to_csv(out_path,index=False)
    sources=dict(base_submission_card['sources'])
    sources['task5']={'model_name':MODEL_NAME,
        'model_card_path':f'task5/save/{MODEL_NAME}.json',
        'model_path':f'task5/save/{MODEL_NAME}.joblib',
        'data_name':DATA_NAME,'data_card_path':f'task5/data/eda/{DATA_NAME}.json',
        'test_path':data_card['splits']['test']['path'],
        'evaluation_metric':'top25_rank_stability_jaccard',
        'evaluation_score':json.loads(eval_path.read_text(encoding='utf8'))['score']}
    detail={'name':SUBMISSION_NAME,'created_at':datetime.now(timezone.utc).isoformat(),
        'card':'Task 5 alert cutoff 25%; Tasks 1-4 identical to scored v6',
        'submission_path':f'submission/{SUBMISSION_NAME}.csv','format':'comma-separated CSV',
        'rows':len(output),'columns':output.columns.tolist(),
        'sha256':sha256(out_path.read_bytes()).hexdigest(),
        'base_submission':BASE_SUBMISSION,'base_sha256':base_submission_card['sha256'],
        'changed_tasks':[5],'task5_changed_rows':int(new_alert.sum()),
        'previous_submission':PRIOR_SUBMISSION,'previous_sha256':prior_card['sha256'],
        'task5_changed_rows_vs_v8':int(new_vs_prior.sum()),
        'task5_alerts':int(candidate_prediction.sum()),
        'task5_metric_note':'Unlabeled rank stability is not macro F3',
        'task5_id_mapping':'Saved EDA Test row position to sample ANS ID, verified against v6',
        'sources':sources}
    out_card_path.write_text(json.dumps(detail,ensure_ascii=False,indent=2)+'\n',encoding='utf8')
saved=pd.read_csv(out_path)
saved_card=json.loads(out_card_path.read_text(encoding='utf8'))
assert saved.equals(output)
assert saved_card['sha256']==sha256(out_path.read_bytes()).hexdigest()
assert saved_card['base_sha256']==base_submission_card['sha256']
print('Prepared Kaggle CSV:',out_path)
print('SHA-256:',saved_card['sha256'])
print('Changed Task 5 rows versus v6:',int(new_alert.sum()),'of',len(output))
print('New Task 5 alerts versus v8:',int(new_vs_prior.sum()))

Prepared Kaggle CSV: C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_t5_alert25_v9.csv
SHA-256: 4d4d251d349dd1414a1db524d647b9187101d140f3ee220fffc724b340ad21f4
Changed Task 5 rows versus v6: 5166 of 25889
New Task 5 alerts versus v8: 2560
